---
title: Steps Towards Refinement Python
date : 2026-04-20
---




Liquid haskell
https://github.com/alcides/aeon
Rocq Program
F*
rosette
grisette
leanette


Shriram paper full monty
esbmc
viper
why3 python
crosshair
deal
icontract


ML framework python overloading
various pytorch 
codon


shorter names

symreflect
symexec
reify
reflect
sdef
smtdef
smt

@symdef
def foo


Nested attributes
foo = bar.biz
foo.baz = 3

This is not gonna do what even I expect.

In [14]:
class Bar():
    pass
x = Bar()
(x.biz, y) = (3,2)
x.biz

3

In [8]:
import ast

print(ast.dump(ast.parse("foo.bar = 3", mode="exec"), indent=4))
print(ast.dump(ast.parse("foo[x] = 3", mode="exec"), indent=4))
print(ast.dump(ast.parse("(foo.bar, biz[3]) = 3", mode="exec"), indent=4))

Module(
    body=[
        Assign(
            targets=[
                Attribute(
                    value=Name(id='foo', ctx=Load()),
                    attr='bar',
                    ctx=Store())],
            value=Constant(value=3))],
    type_ignores=[])
Module(
    body=[
        Assign(
            targets=[
                Subscript(
                    value=Name(id='foo', ctx=Load()),
                    slice=Name(id='x', ctx=Load()),
                    ctx=Store())],
            value=Constant(value=3))],
    type_ignores=[])
Module(
    body=[
        Assign(
            targets=[
                Tuple(
                    elts=[
                        Attribute(
                            value=Name(id='foo', ctx=Load()),
                            attr='bar',
                            ctx=Store()),
                        Subscript(
                            value=Name(id='biz', ctx=Load()),
                            slice=Constant(value=3),
              

In [1]:
from kdrag.all import *

x = Int("x")
A(x, x > 0)


#@symdef
#Expr
Sort

z3.z3types.Sort

In [11]:
from kdrag.all import *
from kdrag.reflect import reflect

@reflect
def myadd(x : int, y : "kd.FinP(x)") -> "kd.FinP(2*x)":
    return x + y
myadd.contract


@reflect
def myabs(x : int) -> "kd.NatP":
    return abs(x)

myabs.contract


@reflect
def safediv(x : int, y : "{x for x in int if x != 0}") -> int:
    return x / y

safediv.contract

       Implies(And(y >= 0, y < x),
               Implies(True,
                       And(myadd(x, y) >= 0,
                           myadd(x, y) < 2*x)))))


In [3]:
foo = kd.define("foo", [], smt.IntVal(42))
@reflect
def bar(x : "{x for x in int if x > 0}") -> "{y for y in int if y == x + 42}":
    () = foo.defn(x)
    return x + 42

ValueError: ('Error reflecting expression', 'foo.defn(x)', "Call(func=Attribute(value=Name(id='foo', ctx=Load()), attr='defn', ctx=Load()), args=[Name(id='x', ctx=Load())], keywords=[])", {'x': x, 'bar': bar})

In [3]:
@reflect
def vapp(n : int, m : int, 
         xs : "smt.ExprRef # kd.Vec(n, kd.Z)", 
         ys : "smt.ExprRef # kd.Vec(m, kd.Z)",
         )->  "smt.ExprRef # kd.Vec(n + m, kd.Z)":
    return xs + ys
vapp.contract

|= ForAll([n, m, xs, ys],
       Implies(And(Length(xs) == n, Length(ys) == m),
               Implies(True,
                       Length(vapp(n, m, xs, ys)) == n + m)))

Ok, should enable [] syntax for Sequences
"#" trick?
Use symunion _inside_ of dictionary. Maybe that would clean things up.
Termination checker
implicits

Use eval to do lookup. This would also find builtins?

keyword arguments for implicits. But then I have to give default? Meh.
type Implicit = smt.ExprRef
type Refine = smt.ExprRef

type Ty = smt.ExprRef . But then it looks like x : Type.
x : "kd.Implicit # "
y : "kd.Refine # "
"_ # "

def elab():

Also contract proving via instantiation. Did I already do this?
A line that evaluates to just kd.Proof puts it in the by?
Hmm.
() = kd.Proof
match also?
Put it in the path cond?
Or should path conds somehow end up in proofs?


class InterpState:
    globals
    locals
    env
    pfs
    path_cond


path sensitivity of proofs
Add in constract instantiations in expr unfortunately

Or should that be a pass in contracts. The interpreter is balooning with complexity

improve and make errors consistent. Emit file line stuff so one can click to it?



termination checker

class Return:
class Locals:
class Impossible:
class AssertFail


In [3]:
import ast
print(ast.dump(ast.parse("() = x"),indent=4))

Module(
    body=[
        Assign(
            targets=[
                Tuple(elts=[], ctx=Store())],
            value=Name(id='x', ctx=Load()))],
    type_ignores=[])


In [4]:
x = smt.Const("x", smt.SeqSort(smt.IntSort()))
x[1:]

AttributeError: 'slice' object has no attribute 'as_ast'

What about a tactic that just instantiates the size definition axioms on all subterms.


In [ ]:
size = SortDispatch("len")
ExprRef.__len__ = size
#x = smt.Int("x")
#size.define([x], smt.Abs(x))
#x = smt.Bool("x")
#size.define([x], smt.If(x, smt.IntVal(1), smt.IntVal(0)))

def default_len(x):
    if isinstance(x, smt.SeqSort):
        return smt.Length(x)
    elif x.sort() == smt.IntSort():
        return smt.Abs(x)
    elif isinstance(x, smt.BitVecRef):
        return smt.BV2Int(e, is_signed=False)


def len_dt(dt : smt.DatatypeSortRef):
    x = smt.Const("x", dt)
    acc = None
    for i in range(dt.num_constructors()):
        constr = dt.constructor(i)
        # or ignore the stuff that doesn't have len?

        body = 1 + smt.Sum(smt.Abs(len(smt.accessor(i,j))) for j in range(constr.arity()))
        if acc is None:
            acc = body
        else:
            smt.If(smt.recognizer(i)(x), body)
    
    kd.define(dt.name() + ".len", [])





In [ ]:
def is_strict_subterm(s : smt.DatatypeRef, t : smt.DatatypeRef) -> smt.BoolRef:
    assert isinstance(s, smt.DatatypeRef) and isinstance(t, smt.DatatypeRef)
    if t.eq(s):
        return smt.BoolVal(False)
    else:
        sort = t.sort()
        todo, down_s = [s], []
        while todo:
            s1 = todo.pop()
            if smt.is_constructor(s1):
                cs = s1.children()
                todo.extend([cs for c in cs if isinstance(c, smt.DatatypeRef)])
                down_s.extend([c for c in cs if c.sort() == sort])
        #return smt.And(t != s, smt.Or([sub_s == t for sub_s in down_s]))
        conds = [sub_s == t for sub_s in down_s]
        down_s.append(s)
        todo, up_t = [t]
        while todo:
            t1 = todo.pop()
            if smt.is_accessor(t1):
                # but we need it guarded by recognizer.
                # find constructor
                x, = t1.children()
                todo.append(x)
                dt = x.sort()
                acc = t1.decl()
                for i in range(dt.num_constructors()):
                    constr = dt.constructor(i)
                    for j in range(constr.arity()):
                        if acc == constr.accessor(i,j):
                            conds.append(smt.Implies(dt.recognizer(i)(x), smt.Or(x == sub_s for sub_s in down_s)))
                else:
                    raise Exception("Didn't find accessor. Something has gone wrong")
            
                t1.argument

# it's too bad this function isn't wired up to python api
def is_subterm(dt : smt.DatatypeSortRef):



In [ ]:
def default_gt(x : smt.ExprRef, y : smt.ExprRef) -> smt.BoolRef:
    if 

In [4]:
import ast
print(ast.dump(ast.parse("x,y,z = (4,5,6)"),indent=4))

Module(
    body=[
        Assign(
            targets=[
                Tuple(
                    elts=[
                        Name(id='x', ctx=Store()),
                        Name(id='y', ctx=Store()),
                        Name(id='z', ctx=Store())],
                    ctx=Store())],
            value=Tuple(
                elts=[
                    Constant(value=4),
                    Constant(value=5),
                    Constant(value=6)],
                ctx=Load()))],
    type_ignores=[])


In [5]:
print(ast.dump(ast.parse("(x,y,z) = (4,5,6)"),indent=4))

Module(
    body=[
        Assign(
            targets=[
                Tuple(
                    elts=[
                        Name(id='x', ctx=Store()),
                        Name(id='y', ctx=Store()),
                        Name(id='z', ctx=Store())],
                    ctx=Store())],
            value=Tuple(
                elts=[
                    Constant(value=4),
                    Constant(value=5),
                    Constant(value=6)],
                ctx=Load()))],
    type_ignores=[])


In [39]:
eval("int", {}, {})

int

In [ ]:
def foo(_x):
    return _x

x = 7
y = 42
def bar():
    x = 3
    z = 4
    print("bar",locals())
    def biz():
        print("biz",locals())
        print(z)
        print("biz x", globals()["x"])
    biz()
bar()


bar {'x': 3, 'z': 4}
biz {'z': 4}
4
biz 7


In [ ]:
type Implicit = smt.ExprRef
type Impl = smt.ExprRef
type _E = smt.ExprRef
type E = smt.ExprRef
type SMT = smt.ExprRef
#type ArgI
#type Id
#type A[T] 


import ast
print(ast.dump(ast.parse("""
def foo(x : "_E # bar", biz : "Implicit # "):
    return x
"""), indent=4))

Module(
    body=[
        FunctionDef(
            name='foo',
            args=arguments(
                posonlyargs=[],
                args=[
                    arg(
                        arg='x',
                        annotation=Constant(value='_E # bar')),
                    arg(
                        arg='biz',
                        annotation=Constant(value='Implicit # '))],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]),
            body=[
                Return(
                    value=Name(id='x', ctx=Load()))],
            decorator_list=[],
            type_params=[])],
    type_ignores=[])


In [ ]:
from kdrag.all import *


    
#def lex_gt(es1, es2):

#def struct_decrease(args, args2):
from typing import Sequence
assert kd.prove(default_measure(smt.IntVal(5)) == 5)
assert default_measure(kd.Nat.S(kd.Nat.Z)) == 2
assert kd.prove(default_measure(kd.seq(1,2,3)) == 3)
assert default_measure(kd.Nat.S(kd.Nat.Z).pred) == 1
x = smt.Const("x", kd.Nat)
assert kd.prove(default_measure(x) > default_measure(x.pred))
assert kd.prove(default_measure(x) > default_measure(x.pred.pred))

y = smt.BitVec("y",32)
kd.prove(smt.BV2Int(y, is_signed=False) >= 0)
kd.prove(default_measure(smt.BitVecVal(-1, 32)) > default_measure(smt.BitVecVal(0, 32)))

def lex_gt(es1 : Sequence[smt.ExprRef], es2 : Sequence[smt.ExprRef]) -> smt.BoolRef:
    assert len(es1) == len(es2)
    return smt.Or(*[smt.And(*[es1[i] == es2[i] for i in range(j)], es1[j] > es2[j]) for j in range(len(es1))])

kd.prove(lex_gt([smt.IntVal(1), smt.IntVal(2)], [smt.IntVal(1), smt.IntVal(1)]))


|= Or(And(1 > 1), And(1 == 1, 2 > 1))

In [ ]:

def all_constraints() -> list[smt.BoolRef]:
    if smt.is_constructor(e):
    elif smt.is_accessor(e):



def wf_gt(x : int, y : int, ctx=by=[]) -> bool:
    assert x.sort() == smt.IntSort()
    assert y.sort() == smt.IntSort()
    return smt.prove(smt.Implies(smt.And(ctx), smt.Abs(x) > smt.Abs(y)), by=by)


In [1]:
from kdrag.all import *
from kdrag.reflect import reflect

@reflect
def mytail(n : int, xs : "kd.Vec(n, smt.IntSort())") -> "kd.Vec(n-1, smt.IntSort())":
    return xs[1:]

ValueError: ('Error reflecting expression', 'xs[1:]', "Subscript(value=Name(id='xs', ctx=Load()), slice=Slice(lower=Constant(value=1)), ctx=Load())", {'n': n, 'xs': xs, 'mytail': mytail})

In [ ]:
type Ty = None
# Ok, so it seems like I can have comments in there? That's fiendish
def foo(x : "smt.ExprRef # Vec(n)") -> "smt.ExprRef # bye":
    return x
